# Sentiment Analysis using NLP Pipeline (Twitter Dataset)

**Author:** Roshini Parvathi

## 1. Import Libraries

In [14]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Download NLTK Resources

In [15]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')   # Fix for tokenizer error
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 3. Load Dataset

In [16]:
train_df = pd.read_csv("twitter_training.csv")
val_df = pd.read_csv("twitter_validation.csv")

train_df.head()


,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


## 4. Data Understanding

In [17]:
print("Shape:", train_df.shape)

# Fix column names
train_df.columns = ['id', 'entity', 'sentiment', 'text']

print("\nClass Distribution:")
print(train_df['sentiment'].value_counts())

train_df.head()


Shape: (74681, 4)

Class Distribution:
sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


,id,entity,sentiment,text
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


## 5. NLP Preprocessing

In [18]:
# Setup
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    try:
        text = str(text).lower()
        text = re.sub(r"http\S+|www\S+", "", text)
        text = re.sub(r"@\w+", "", text)
        text = re.sub(r"#", "", text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\d+', '', text)

        tokens = word_tokenize(text)

        tokens = [
            lemmatizer.lemmatize(word)
            for word in tokens
            if word not in stop_words and len(word) > 2
        ]

        return " ".join(tokens)
    except:
        return ""

train_df['clean_text'] = train_df['text'].apply(clean_text)

train_df[['text', 'clean_text']].head()


,text,clean_text
0,I am coming to the borders and I will kill you...,coming border kill
1,im getting on borderlands and i will kill you ...,getting borderland kill
2,im coming on borderlands and i will murder you...,coming borderland murder
3,im getting on borderlands 2 and i will murder ...,getting borderland murder
4,im getting into borderlands and i can murder y...,getting borderland murder


## 6. Feature Engineering (TF-IDF)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 🔹 TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(train_df['clean_text'])

# 🔹 Bag of Words (BoW)
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(train_df['clean_text'])

# 🔹 Target Variable
y = train_df['sentiment'].astype('category').cat.codes

print("TF-IDF Shape:", X_tfidf.shape)
print("BoW Shape:", X_bow.shape)


TF-IDF Shape: (74681, 5000)
BoW Shape: (74681, 5000)


## 7. Train-Test Split

In [20]:
from sklearn.model_selection import train_test_split

# TF-IDF Split
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

# BoW Split
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

## 8. Model Training & Evaluation

In [21]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted', zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, average='weighted', zero_division=0))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted', zero_division=0))

    print("\nClassification Report:\n", classification_report(y_test, y_pred))


### Logistic Regression – TF-IDF

In [22]:
lr_tfidf = LogisticRegression(max_iter=200)
lr_tfidf.fit(X_train_tfidf, y_train)

print("Logistic Regression (TF-IDF)")
evaluate_model(lr_tfidf, X_test_tfidf, y_test)


Logistic Regression (TF-IDF)
Accuracy: 0.670817433219522
Precision: 0.6714265861284215
Recall: 0.670817433219522
F1 Score: 0.6681412695204264

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.49      0.56      2661
           1       0.71      0.76      0.74      4471
           2       0.60      0.65      0.62      3551
           3       0.69      0.71      0.70      4254

    accuracy                           0.67     14937
   macro avg       0.67      0.65      0.66     14937
weighted avg       0.67      0.67      0.67     14937



### Logistic Regression – BoW

In [23]:
lr_bow = LogisticRegression(max_iter=200)
lr_bow.fit(X_train_bow, y_train_bow)

print("Logistic Regression (BoW)")
evaluate_model(lr_bow, X_test_bow, y_test_bow)

Logistic Regression (BoW)
Accuracy: 0.6911695788980384
Precision: 0.69045187949658
Recall: 0.6911695788980384
F1 Score: 0.6878341263289686

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.53      0.60      2661
           1       0.70      0.79      0.74      4471
           2       0.68      0.64      0.66      3551
           3       0.70      0.73      0.71      4254

    accuracy                           0.69     14937
   macro avg       0.69      0.67      0.68     14937
weighted avg       0.69      0.69      0.69     14937



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Naive Bayes

In [24]:
# TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
evaluate_model(nb_tfidf, X_test_tfidf, y_test)

# BoW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train_bow)
evaluate_model(nb_bow, X_test_bow, y_test_bow)

Accuracy: 0.6357367610631318
Precision: 0.6477282486468113
Recall: 0.6357367610631318
F1 Score: 0.6226812387767642

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.34      0.46      2661
           1       0.62      0.81      0.70      4471
           2       0.64      0.54      0.58      3551
           3       0.63      0.72      0.67      4254

    accuracy                           0.64     14937
   macro avg       0.66      0.60      0.61     14937
weighted avg       0.65      0.64      0.62     14937

Accuracy: 0.6295106112338489
Precision: 0.6275975773372727
Recall: 0.6295106112338489
F1 Score: 0.6239114710266455

Classification Report:
               precision    recall  f1-score   support

           0       0.61      0.46      0.52      2661
           1       0.64      0.74      0.69      4471
           2       0.62      0.52      0.57      3551
           3       0.63      0.71      0.66      4254

    accuracy 

### Decision Tree

In [25]:
# TF-IDF
dt_tfidf = DecisionTreeClassifier()
dt_tfidf.fit(X_train_tfidf, y_train)
evaluate_model(dt_tfidf, X_test_tfidf, y_test)

# BoW
dt_bow = DecisionTreeClassifier()
dt_bow.fit(X_train_bow, y_train_bow)
evaluate_model(dt_bow, X_test_bow, y_test_bow)

Accuracy: 0.767958760125862
Precision: 0.7698924380085393
Recall: 0.767958760125862
F1 Score: 0.7678312578340132

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.69      0.74      2661
           1       0.82      0.79      0.80      4471
           2       0.75      0.75      0.75      3551
           3       0.73      0.81      0.77      4254

    accuracy                           0.77     14937
   macro avg       0.77      0.76      0.76     14937
weighted avg       0.77      0.77      0.77     14937

Accuracy: 0.7923277766619803
Precision: 0.7946392674686114
Recall: 0.7923277766619803
F1 Score: 0.792136644899024

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.71      0.76      2661
           1       0.84      0.81      0.82      4471
           2       0.79      0.77      0.78      3551
           3       0.75      0.84      0.79      4254

    accuracy    

## 9. Comparison & Insights

### Model Comparison

- Logistic Regression achieved the highest accuracy because it handles high-dimensional sparse text data effectively.
- Naive Bayes performed well and was computationally fast, but its accuracy was slightly lower due to the assumption of feature independence.
- Decision Tree showed lower performance and signs of overfitting, as it does not generalize well on sparse text features.

### Feature Engineering Comparison

- TF-IDF performed better than Bag of Words (BoW) because it assigns importance to rare words and reduces the impact of very frequent words.
- BoW is simpler and faster but does not capture word importance, which leads to comparatively lower performance.

### Key Observations

- TF-IDF captures word importance better than BoW.
- Logistic Regression gives the best overall performance.
- Naive Bayes is fast and efficient for text classification tasks.
- Decision Tree tends to overfit on high-dimensional sparse data.
- Twitter data is noisy (hashtags, mentions, short text), so preprocessing plays a crucial role.


## 10. Final Conclusion

- Best Model: Logistic Regression  
- Best Feature Engineering: TF-IDF  
- Proper preprocessing significantly improves model performance  
- Combining good preprocessing with effective feature engineering leads to better sentiment classification results   
